# Cubierta Forestal · Experimentación de ML (notebook autónomo)

Flujo reproducible de Machine Learning: **PostgreSQL (`training.covertype_features`) → Gold Layer →
EDA → Feature Engineering → Train/Val/Test → Modelo único (AutoGluon) → 25 configuraciones →
25 experimentos → métricas → tabla consolidada → preparado para MLflow**.

**Fuente de datos:** los datos ya están cargados en PostgreSQL; el notebook los lee de la tabla
**`training.covertype_features`** (snapshot versionado con la columna `split` ya decidida). Dentro
del contenedor `jupyter` del `docker-compose` la conexión funciona sin configurar nada (las
variables `DATA_DB_*` ya vienen inyectadas). Como respaldo, si no hubiera base disponible, el
notebook puede descargar el dataset oficial con `sklearn.datasets.fetch_covtype` para poder correr
igualmente fuera del stack.

**Integra MLflow (S2):** cada una de las 25 configuraciones se registra como un *run* y el mejor
modelo se publica en el Model Registry para que la Inference API lo sirva. Si no hay servidor
MLflow, el notebook corre igual y deja los payloads locales. **Fuera de alcance:** la API de
inferencia (S6) y el despliegue; MinIO (S3) sólo se usa indirectamente como almacén de artefactos
del servidor MLflow.

## 01. Configuración del entorno

In [ ]:
from pathlib import Path

SEED = 42
N_CONFIGS = 25

ARTIFACTS_DIR = Path("artifacts").resolve()
MODELS_DIR = ARTIFACTS_DIR / "models"
RESULTS_DIR = ARTIFACTS_DIR / "results"
MLFLOW_DIR = ARTIFACTS_DIR / "mlflow_payloads"
for d in (MODELS_DIR, RESULTS_DIR, MLFLOW_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("SEED =", SEED, "| N_CONFIGS =", N_CONFIGS, "| artifacts ->", ARTIFACTS_DIR)

In [ ]:
# Bootstrap idempotente de dependencias (para que corra desde cero, p.ej. en VS Code).
import importlib, subprocess, sys

def _ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name); return False
    except ImportError:
        print(f"instalando {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pip_name]); return True

_ensure("autogluon.tabular[lightgbm]", "autogluon.tabular")   # núcleo del experimento
for _p, _i in [("scikit-learn", "sklearn"), ("pandas", "pandas"), ("pyarrow", "pyarrow"),
               ("matplotlib", "matplotlib"), ("seaborn", "seaborn"),
               ("SQLAlchemy", "sqlalchemy"), ("psycopg[binary]", "psycopg"),
               ("mlflow", "mlflow"), ("python-dotenv", "dotenv")]:
    try: _ensure(_p, _i)
    except Exception as exc:  # noqa: BLE001
        print(f"aviso: no se pudo instalar {_p}: {exc}")
print("bootstrap listo")

## 02. Importación de librerías

In [ ]:
import io, json, os, time, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from autogluon.tabular import TabularPredictor
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 140)
np.random.seed(SEED)
print("librerías importadas")

## 03. Configuración de variables de entorno

**Sin credenciales en el código.** Si vas a usar una base PostgreSQL, define estas variables (o un
archivo `.env`, ver `.env.example`). **Si no hay base, no pasa nada**: el notebook descarga
Covertype automáticamente en la sección 05.

```dotenv
DATA_DB_HOST=localhost
DATA_DB_PORT=5432
DATA_DB_NAME=cubierta_forestal
DATA_DB_USER=app
DATA_DB_PASSWORD=app
```

In [ ]:
try:
    from dotenv import find_dotenv, load_dotenv
    _d = find_dotenv(usecwd=True)
    if _d: load_dotenv(_d, override=False); print(".env cargado:", _d)
    else: print("no hay .env; se usan variables de entorno del proceso (o fallback a fetch_covtype)")
except Exception as exc:  # noqa: BLE001
    print("python-dotenv no disponible:", exc)

# Defaults iguales al docker-compose del taller (servicio jupyter): dentro del contenedor
# las variables de entorno ya vienen inyectadas (DATA_DB_HOST=postgres, etc.) y tienen prioridad.
DB_CONFIG = {
    "host": os.environ.get("DATA_DB_HOST", "localhost"),
    "port": os.environ.get("DATA_DB_PORT", "5432"),
    "dbname": os.environ.get("DATA_DB_NAME", "cubierta_forestal"),
    "user": os.environ.get("DATA_DB_USER", "app"),
    "password": os.environ.get("DATA_DB_PASSWORD", "app"),
}
# Pin opcional de versión del dataset (si no, se toma la más reciente de training.dataset_version).
DATASET_VERSION_PIN = os.environ.get("DATASET_VERSION", "").strip()
USE_DB = bool(DB_CONFIG["host"] and DB_CONFIG["dbname"] and DB_CONFIG["user"])
print("¿intentar PostgreSQL?", USE_DB, "|", {k: v for k, v in DB_CONFIG.items() if k != "password"})

## 04. Conexión a PostgreSQL (opcional)

Función reutilizable `connect_to_postgresql()` sobre SQLAlchemy + psycopg v3. Sólo se usa si hay
configuración de base; si falla o no hay base, el notebook sigue con el dataset oficial.

In [ ]:
def connect_to_postgresql(config: dict = None, driver: str = "psycopg", echo: bool = False):
    '''Crea y valida un SQLAlchemy Engine. Lanza excepción si no puede conectar.'''
    from sqlalchemy import create_engine, text
    config = config or DB_CONFIG
    url = (f"postgresql+{driver}://{config['user']}:{config['password']}"
           f"@{config['host']}:{config['port']}/{config['dbname']}")
    engine = create_engine(url, pool_pre_ping=True, echo=echo)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    return engine


engine = None
if USE_DB:
    try:
        engine = connect_to_postgresql()
        print("conexión a PostgreSQL OK")
    except Exception as exc:  # noqa: BLE001
        print(f"no se pudo conectar a PostgreSQL ({exc}); se usará fetch_covtype")
        engine = None
else:
    print("sin configuración de base: se usará fetch_covtype")

## 05. Exploración / carga de la fuente de datos

**Fuente principal: `training.covertype_features`** — los datos ya están cargados y listos en
PostgreSQL (snapshot versionado, con la columna `split` train/validation/test ya decidida). El
notebook los toma de ahí.

`load_covertype()` resuelve la fuente en este orden:

1. **`training.covertype_features`** (fuente oficial, ya cargada). Toma la versión indicada en
   `DATASET_VERSION`, o la más reciente de `training.dataset_version`. Trae también `split`.
2. **`processed.covertype`** si por algún motivo no hubiera snapshot versionado.
3. **`sklearn.datasets.fetch_covtype`** sólo como último recurso si no hay base disponible
   (reconstruyendo el esquema del proyecto), para poder ejecutar el notebook fuera del stack.

Resultado: un DataFrame con el **contrato de features** (10 numéricas + 2 categóricas + target).

In [ ]:
NUMERIC = ["elevation", "aspect", "slope",
           "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
           "horizontal_distance_to_roadways",
           "hillshade_9am", "hillshade_noon", "hillshade_3pm",
           "horizontal_distance_to_fire_points"]
CATEGORICAL = ["wilderness_area", "soil_type"]
TARGET = "cover_type"
CONTRACT = NUMERIC + CATEGORICAL + [TARGET]


def _load_from_sklearn():
    from sklearn.datasets import fetch_covtype
    frame = fetch_covtype(as_frame=True).frame
    num_map = {
        "Elevation": "elevation", "Aspect": "aspect", "Slope": "slope",
        "Horizontal_Distance_To_Hydrology": "horizontal_distance_to_hydrology",
        "Vertical_Distance_To_Hydrology": "vertical_distance_to_hydrology",
        "Horizontal_Distance_To_Roadways": "horizontal_distance_to_roadways",
        "Hillshade_9am": "hillshade_9am", "Hillshade_Noon": "hillshade_noon",
        "Hillshade_3pm": "hillshade_3pm",
        "Horizontal_Distance_To_Fire_Points": "horizontal_distance_to_fire_points",
    }
    df = frame.rename(columns=num_map)
    wild = [c for c in frame if c.startswith("Wilderness_Area_")]
    soil = [c for c in frame if c.startswith("Soil_Type_")]
    df["wilderness_area"] = frame[wild].idxmax(axis=1).str.replace("Wilderness_Area_", "wilderness_", regex=False)
    df["soil_type"] = frame[soil].idxmax(axis=1).str.replace("Soil_Type_", "soil_", regex=False)
    df["cover_type"] = frame["Cover_Type"].astype(int) - 1  # 0..6
    return df[CONTRACT].copy(), "sklearn.fetch_covtype"


def load_covertype():
    # (1) Fuente principal: training.covertype_features (ya cargada, con split).
    if engine is not None:
        from sqlalchemy import text
        try:
            if DATASET_VERSION_PIN:
                ver = DATASET_VERSION_PIN
            else:
                v = pd.read_sql(text("SELECT version FROM training.dataset_version ORDER BY created_at DESC LIMIT 1"), engine)
                ver = v.iloc[0]["version"] if not v.empty else None
            if ver is not None:
                df = pd.read_sql(
                    text(f"SELECT {', '.join(CONTRACT)}, split FROM training.covertype_features WHERE dataset_version = :v"),
                    engine, params={"v": ver})
                if not df.empty:
                    return df, f"postgres:training.covertype_features@{ver}"
                print(f"training.covertype_features sin filas para la versión '{ver}'")
        except Exception as exc:  # noqa: BLE001
            print("training.covertype_features no disponible:", exc)
        # (2) Respaldo dentro de la base.
        try:
            df = pd.read_sql(text(f"SELECT {', '.join(CONTRACT)} FROM processed.covertype"), engine)
            if not df.empty:
                return df, "postgres:processed.covertype"
        except Exception as exc:  # noqa: BLE001
            print("processed.covertype no disponible:", exc)
    # (3) Último recurso fuera del stack.
    return _load_from_sklearn()


raw_df, DATA_SOURCE = load_covertype()
print("fuente de datos:", DATA_SOURCE, "| shape:", raw_df.shape)
raw_df.head()

## 06. Validación de calidad

In [ ]:
quality = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "n_null": raw_df.isna().sum(),
    "pct_null": (raw_df.isna().mean() * 100).round(3),
    "n_unique": raw_df.nunique(),
})
display(quality)

dups = raw_df.duplicated(subset=CONTRACT).sum()
print("duplicados exactos del vector de features:", dups)
display(raw_df.describe(include=[np.number]).T)
display(raw_df.describe(include=["object"]).T)

## 07. Construcción / validación de Gold Layer

**Gold Layer** = dataset analítico limpio, tipado y consistente para ML. Transformaciones
(trazabilidad **Fuente → Gold → ML**):

- tipado correcto, normalización de texto de las categóricas (minúsculas, sin espacios),
- descarte de nulos y duplicados exactos,
- recorte de valores fuera de rango (reflejando los rangos físicos del dataset),
- **split `train/validation/test`** determinista y reproducible (se respeta el de la base si vino
  en la columna `split`; si no, se crea por hash estable por fila → sin fuga del test).

In [ ]:
import hashlib

gold_df = raw_df.copy()
# ¿viene el split predefinido de training.covertype_features? La columna viaja con cada fila.
has_db_split = "split" in gold_df.columns
# normalización de categóricas
for c in CATEGORICAL:
    gold_df[c] = gold_df[c].astype(str).str.strip().str.lower()
# tipado
for c in NUMERIC:
    gold_df[c] = pd.to_numeric(gold_df[c], errors="coerce")
gold_df[TARGET] = pd.to_numeric(gold_df[TARGET], errors="coerce").astype("Int64")
# nulos + duplicados exactos (si los datos vienen de la Gold de la base, normalmente no hay)
gold_df = gold_df.dropna(subset=CONTRACT).drop_duplicates(subset=CONTRACT).reset_index(drop=True)
gold_df[TARGET] = gold_df[TARGET].astype(int)
# recorte de rangos físicos
RANGES = {"aspect": (0, 360), "slope": (0, 90),
          "hillshade_9am": (0, 255), "hillshade_noon": (0, 255), "hillshade_3pm": (0, 255),
          "cover_type": (0, 6)}
for col, (lo, hi) in RANGES.items():
    gold_df = gold_df[gold_df[col].between(lo, hi)]
for col in [c for c in gold_df.columns if c.startswith("horizontal_distance")]:
    gold_df = gold_df[gold_df[col] >= 0]
gold_df = gold_df.reset_index(drop=True)

# split: se RESPETA el de training.covertype_features (la columna ya viajó con cada fila).
# Sólo se genera uno determinista si la fuente no traía split (p. ej. fallback a fetch_covtype).
if has_db_split and not gold_df["split"].isna().any():
    SPLIT_ORIGIN = "training.covertype_features (predefinido)"
else:
    u = gold_df.index.to_series().map(lambda i: int(hashlib.md5(f"{i}:{SEED}".encode()).hexdigest()[:8], 16) / 0xFFFFFFFF)
    gold_df["split"] = np.where(u < 0.70, "train", np.where(u < 0.85, "validation", "test"))
    SPLIT_ORIGIN = "determinista por hash (no venía split en la fuente)"

DATASET_VERSION = DATA_SOURCE
print("gold_df:", gold_df.shape, "| origen del split:", SPLIT_ORIGIN)
display(gold_df["split"].value_counts())

# validación
checks = {
    "sin_nulos": bool(gold_df.drop(columns=["split"]).isna().sum().sum() == 0),
    "target_presente": TARGET in gold_df.columns,
    "splits_ok": set(gold_df["split"].unique()) <= {"train", "validation", "test"},
    "rangos_ok": bool(all(gold_df[c].between(lo, hi).all() for c, (lo, hi) in RANGES.items())),
}
print(checks); assert all(checks.values()), checks
print("Gold Layer validada ✔")

## 08. Definición del problema

- **Variable objetivo:** `cover_type` (0–6) — tipo de cubierta forestal de cada parcela de 30×30 m.
- **Tipo de problema:** **clasificación multiclase** (7 clases).
- **Predictores:** 10 numéricas + 2 categóricas (`wilderness_area`, `soil_type`).
- **Descartadas:** cualquier identificador/linaje (no presente en el contrato) para evitar leakage.
- **Nota geográfica:** `wilderness_area` es localización; predictor válido, pero las parcelas están
  autocorrelacionadas espacialmente, por lo que un split aleatorio puede filtrar información entre
  celdas vecinas. Alternativa más conservadora: validación por grupos (`GroupKFold` por
  `wilderness_area`). Aquí se usa el split determinista por reproducibilidad.

In [ ]:
target_counts = gold_df[TARGET].value_counts().sort_index()
display(pd.DataFrame({"n": target_counts, "pct": (target_counts / len(gold_df) * 100).round(2)}))
print("clases:", gold_df[TARGET].nunique(),
      "| ratio desbalance:", round(target_counts.max() / target_counts.min(), 1), "x")
PROBLEM_TYPE = "multiclass"

## 09. Análisis exploratorio (EDA)

In [ ]:
print("resumen numérico:"); display(gold_df[NUMERIC].describe().T)
for c in CATEGORICAL:
    print(f"{c}: {gold_df[c].nunique()} categorías")
corr = gold_df[NUMERIC].corr(numeric_only=True).round(2)
display(corr)
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt, seaborn as sns
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    sns.heatmap(corr, cmap="viridis", ax=ax[0]); ax[0].set_title("Correlación (numéricas)")
    gold_df[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax[1]); ax[1].set_title("Target")
    plt.tight_layout(); fig.savefig(ARTIFACTS_DIR / "eda_overview.png", dpi=90); plt.show()
except Exception as exc:  # noqa: BLE001
    print("(gráficos omitidos:", exc, ")")

## 10. Feature Engineering

Transformaciones deterministas fila a fila (sin fit → sin leakage): tipado de categóricas a
`category` y feature derivada `euclidean_distance_to_hydrology` = √(horizontal² + vertical²).
La codificación de categóricas la hace AutoGluon internamente (viaja con el modelo).

In [ ]:
FEATURE_ENGINEERING = True

def engineer(df):
    out = df.copy()
    for c in CATEGORICAL: out[c] = out[c].astype("category")
    if FEATURE_ENGINEERING:
        out["euclidean_distance_to_hydrology"] = np.sqrt(
            out["horizontal_distance_to_hydrology"].astype(float) ** 2
            + out["vertical_distance_to_hydrology"].astype(float) ** 2).round(3)
    return out

gold_fe = engineer(gold_df)
FEATURES = NUMERIC + CATEGORICAL + (["euclidean_distance_to_hydrology"] if FEATURE_ENGINEERING else [])
print(len(FEATURES), "features:", FEATURES)
gold_fe.head()

## 11. Train / Validation / Test

Se respeta el split ya decidido. **El TEST permanece aislado**: no se usa para entrenar, comparar
ni seleccionar; sólo se evalúa una vez al final sobre la configuración ganadora.

In [ ]:
def split_xy(df, s):
    part = df[df["split"] == s]
    return part[FEATURES].reset_index(drop=True), part[TARGET].reset_index(drop=True)

X_train, y_train = split_xy(gold_fe, "train")
X_val, y_val = split_xy(gold_fe, "validation")
X_test, y_test = split_xy(gold_fe, "test")
train_data = X_train.copy(); train_data[TARGET] = y_train

DATASET_SIZES = {"train": len(X_train), "validation": len(X_val), "test": len(X_test)}
N_FEATURES = len(FEATURES); CLASSES = sorted(gold_fe[TARGET].unique().tolist())
print("tamaños:", DATASET_SIZES, "| features:", N_FEATURES, "| clases:", CLASSES)
assert DATASET_SIZES["test"] > 0 and DATASET_SIZES["train"] > 0
print("TEST aislado ✔")

## 12. Definición del modelo

**Una sola familia:** gradient-boosted trees (**LightGBM**, clave `GBM` en AutoGluon). Elegida por
el tipo de problema (multiclase tabular), la mezcla numérica+categórica, el tamaño del dataset, su
rapidez (factible correr 25 experimentos), interpretabilidad y compatibilidad con MLflow.
Intercambiable cambiando `MODEL_FAMILY` (p.ej. `"XGB"`, `"CAT"`, `"RF"`).

In [ ]:
MODEL_FAMILY = "GBM"
AG_EVAL_METRIC = "f1_macro"
print("familia:", MODEL_FAMILY, "| eval_metric:", AG_EVAL_METRIC)

## 13. Diseño de las 25 configuraciones

Espacio justificado para LightGBM: **5 learning rates × 5 niveles de capacidad/regularización = 25**
configuraciones que barren zonas distintas (no variaciones mínimas). HPO en malla determinista
(no la búsqueda interna de AutoGluon) para tener 25 experimentos reproducibles e identificables.

In [ ]:
LEARNING_RATES = [0.01, 0.03, 0.05, 0.1, 0.2]
CAPACITY_TIERS = [
    {"num_leaves": 15,  "max_depth": 4,  "num_boost_round": 200,  "min_data_in_leaf": 60,
     "feature_fraction": 0.60, "bagging_fraction": 0.60, "bagging_freq": 1, "lambda_l1": 1.0, "lambda_l2": 1.0},
    {"num_leaves": 31,  "max_depth": 6,  "num_boost_round": 400,  "min_data_in_leaf": 40,
     "feature_fraction": 0.70, "bagging_fraction": 0.70, "bagging_freq": 1, "lambda_l1": 0.5, "lambda_l2": 0.5},
    {"num_leaves": 63,  "max_depth": 8,  "num_boost_round": 600,  "min_data_in_leaf": 20,
     "feature_fraction": 0.80, "bagging_fraction": 0.80, "bagging_freq": 1, "lambda_l1": 0.1, "lambda_l2": 0.1},
    {"num_leaves": 127, "max_depth": 12, "num_boost_round": 800,  "min_data_in_leaf": 10,
     "feature_fraction": 0.90, "bagging_fraction": 0.90, "bagging_freq": 1, "lambda_l1": 0.0, "lambda_l2": 0.05},
    {"num_leaves": 255, "max_depth": -1, "num_boost_round": 1200, "min_data_in_leaf": 5,
     "feature_fraction": 1.00, "bagging_fraction": 1.00, "bagging_freq": 0, "lambda_l1": 0.0, "lambda_l2": 0.0},
]

def build_configurations(seed=SEED, model_family=MODEL_FAMILY):
    configs, idx = [], 1
    stamp = datetime.now(timezone.utc).isoformat()
    for lr in LEARNING_RATES:
        for tier in CAPACITY_TIERS:
            configs.append({"config_id": f"config_{idx:02d}", "experiment": f"Experiment {idx:02d}",
                            "model": model_family, "hyperparameters": {"learning_rate": lr, "seed": seed, **tier},
                            "seed": seed, "timestamp": stamp}); idx += 1
    return configs

configurations = build_configurations()
assert len(configurations) == N_CONFIGS
assert len({json.dumps(c["hyperparameters"], sort_keys=True) for c in configurations}) == N_CONFIGS
json.dump(configurations, open(ARTIFACTS_DIR / "configurations.json", "w"), indent=2)
print(len(configurations), "configuraciones únicas")
pd.json_normalize(configurations).head()

## 13b. Conexión con MLflow (S2)

Jupyter (S4) se conecta con MLflow (S2) para registrar cada experimento como un *run*. Se activa
automáticamente si hay un `MLFLOW_TRACKING_URI` disponible (dentro del `docker-compose` del taller
ya viene inyectado: `http://mlflow:5000`). Si no hay servidor, el notebook **sigue corriendo** y
sólo deja los payloads locales — así nunca falla fuera del stack.

Por cada run se registran `params`, `metrics`, `tags` y el modelo (artefacto del `TabularPredictor`).
El **mejor** modelo se publica en el **Model Registry** como `MODEL_NAME` con el alias `MODEL_ALIAS`
(`production`), que es justo lo que la Inference API (S6) carga: `models:/<MODEL_NAME>@<MODEL_ALIAS>`.

In [ ]:
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "").strip()
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", "cubierta-forestal")
MODEL_NAME = os.environ.get("MODEL_NAME", "cubierta-forestal")   # nombre en el Model Registry
MODEL_ALIAS = os.environ.get("MODEL_ALIAS", "production")        # alias que sirve la Inference API
LOG_MODEL_EACH_RUN = True   # registrar el modelo de cada run (False = solo params/métricas)

USE_MLFLOW, mlflow = False, None
if MLFLOW_TRACKING_URI:
    try:
        import mlflow as _mlflow
        _mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        _mlflow.set_experiment(MLFLOW_EXPERIMENT)
        mlflow = _mlflow
        USE_MLFLOW = True
        print(f"MLflow conectado: {MLFLOW_TRACKING_URI} | experimento '{MLFLOW_EXPERIMENT}'")
    except Exception as exc:  # noqa: BLE001
        print(f"MLflow no disponible ({exc}); se continúa sin registrar (solo payloads locales)")
else:
    print("MLFLOW_TRACKING_URI no definido; se continúa sin MLflow (solo payloads locales)")


# Envoltura pyfunc para servir el TabularPredictor de AutoGluon desde el Model Registry.
if USE_MLFLOW:
    class AutogluonTabularModel(mlflow.pyfunc.PythonModel):
        def load_context(self, context):
            from autogluon.tabular import TabularPredictor
            self._predictor = TabularPredictor.load(context.artifacts["predictor"])

        def predict(self, context, model_input):
            return self._predictor.predict(model_input)

RUN_IDS = {}   # config_id -> mlflow run_id (para registrar luego el mejor)

## 14. Ejecución de los 25 experimentos

Cada configuración entrena un `TabularPredictor` independiente con **un solo modelo GBM** (sin
bagging/stacking/ensemble). Se guarda en `models/config_XX/`, se registran params, métricas de
validación, tiempo, tamaños, nº features y semilla, y —si MLflow está activo— **cada configuración
se abre como un run** (`mlflow.start_run`) donde se loguean params, métricas y el modelo.
**El TEST no se toca aquí.**

In [ ]:
def compute_metrics(y_true, y_pred, y_proba=None, labels=None, prefix=""):
    m = {f"{prefix}accuracy": accuracy_score(y_true, y_pred),
         f"{prefix}balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
         f"{prefix}f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
         f"{prefix}f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
         f"{prefix}precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
         f"{prefix}recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0)}
    if y_proba is not None:
        try:
            m[f"{prefix}log_loss"] = log_loss(y_true, y_proba, labels=labels)
            m[f"{prefix}roc_auc_ovr_macro"] = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro", labels=labels)
        except Exception: pass
    return {k: float(round(v, 6)) for k, v in m.items()}

In [ ]:
PRIMARY_METRIC = "val_f1_macro"
QUICK_SMOKE = False   # True = subconjunto pequeño para probar rápido
_train = train_data.sample(n=min(20000, len(train_data)), random_state=SEED) if QUICK_SMOKE else train_data

records = []
for cfg in configurations:
    cid = cfg["config_id"]; mp = MODELS_DIR / cid
    run = mlflow.start_run(run_name=cfg["experiment"]) if USE_MLFLOW else None
    try:
        t0 = time.perf_counter()
        predictor = TabularPredictor(label=TARGET, problem_type=PROBLEM_TYPE,
                                     eval_metric=AG_EVAL_METRIC, path=str(mp), verbosity=0)
        predictor.fit(train_data=_train, hyperparameters={MODEL_FAMILY: cfg["hyperparameters"]},
                      num_bag_folds=0, num_stack_levels=0, fit_weighted_ensemble=False)
        train_time = round(time.perf_counter() - t0, 3)

        val_pred = predictor.predict(X_val)
        try: val_proba = predictor.predict_proba(X_val)[CLASSES].to_numpy()
        except Exception: val_proba = None
        vm = compute_metrics(y_val, val_pred, val_proba, labels=CLASSES, prefix="val_")

        rec = {"config_id": cid, "experiment": cfg["experiment"], "model": cfg["model"],
               "seed": cfg["seed"], "timestamp": cfg["timestamp"], "train_time_s": train_time,
               "n_train": DATASET_SIZES["train"], "n_validation": DATASET_SIZES["validation"],
               "n_test": DATASET_SIZES["test"], "n_features": N_FEATURES,
               "dataset_version": DATASET_VERSION, "model_path": str(mp),
               **{f"param_{k}": v for k, v in cfg["hyperparameters"].items()}, **vm}
        records.append(rec)

        # Payload local (siempre, para trazabilidad fuera de MLflow).
        json.dump({"run_name": cfg["experiment"],
                   "params": {**cfg["hyperparameters"], "model_family": cfg["model"],
                              "dataset_version": DATASET_VERSION, "n_features": N_FEATURES},
                   "metrics": vm,
                   "artifacts": {"model_dir": str(mp), "config_json": str(ARTIFACTS_DIR / "configurations.json")},
                   "tags": {"seed": cfg["seed"], "problem_type": PROBLEM_TYPE}},
                  open(MLFLOW_DIR / f"{cid}.json", "w"), indent=2)

        # Registro en MLflow (si está activo).
        if USE_MLFLOW:
            RUN_IDS[cid] = run.info.run_id
            mlflow.log_params({**cfg["hyperparameters"], "model_family": cfg["model"],
                               "dataset_version": DATASET_VERSION, "n_features": N_FEATURES})
            mlflow.log_metrics(vm)
            mlflow.set_tags({"config_id": cid, "seed": cfg["seed"], "problem_type": PROBLEM_TYPE,
                             "train_time_s": train_time, "model_family": cfg["model"]})
            try:
                mlflow.log_artifact(str(ARTIFACTS_DIR / "configurations.json"))
                if LOG_MODEL_EACH_RUN:
                    mlflow.log_artifacts(str(mp), artifact_path="autogluon_predictor")
            except Exception as exc:  # noqa: BLE001
                print(f"  aviso: no se pudo subir artefacto de {cid}: {exc}")

        print(f"{cfg['experiment']:<14} {PRIMARY_METRIC}={rec[PRIMARY_METRIC]:.4f}  ({train_time}s)"
              + ("  [mlflow ✓]" if USE_MLFLOW else ""))
    finally:
        if run is not None:
            mlflow.end_run()
print("\n25 experimentos completados" + (" y registrados en MLflow" if USE_MLFLOW else ""))

## 15. Evaluación

**Métrica principal:** `f1_macro` en validation (dataset desbalanceado → pondera por igual las 7
clases). Se reportan además accuracy, balanced_accuracy, precisión/recall macro, log_loss y
ROC-AUC (OVR macro). Matriz de confusión sólo para la ganadora.

In [ ]:
results_df = pd.DataFrame(records)
best_row = results_df.sort_values(PRIMARY_METRIC, ascending=False).iloc[0]
BEST_CONFIG_ID = best_row["config_id"]
print("mejor:", BEST_CONFIG_ID, "=", round(best_row[PRIMARY_METRIC], 4))

best_predictor = TabularPredictor.load(str(MODELS_DIR / BEST_CONFIG_ID))
val_pred_best = best_predictor.predict(X_val)
cm = confusion_matrix(y_val, val_pred_best, labels=CLASSES)
display(pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES], columns=[f"pred_{c}" for c in CLASSES]))
print(classification_report(y_val, val_pred_best, zero_division=0))

## 16. Consolidación de resultados

In [ ]:
assert len(results_df) == N_CONFIGS, len(results_df)
col_order = (["config_id", "experiment", "model", "dataset_version", "n_train", "n_validation",
              "n_test", "n_features", "seed", "train_time_s"]
             + [c for c in results_df.columns if c.startswith("param_")]
             + [c for c in results_df.columns if c.startswith("val_")]
             + ["model_path", "timestamp"])
results_df = results_df[[c for c in col_order if c in results_df.columns]]
print("results_df:", results_df.shape); results_df.head()

## 17. Comparación de las 25 configuraciones

In [ ]:
ranking = results_df.sort_values(PRIMARY_METRIC, ascending=False)[
    ["config_id", "param_learning_rate", "param_num_leaves", "param_num_boost_round",
     PRIMARY_METRIC, "val_accuracy", "val_balanced_accuracy", "train_time_s"]].reset_index(drop=True)
display(ranking)
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(results_df["config_id"], results_df[PRIMARY_METRIC])
    ax.set_xticklabels(results_df["config_id"], rotation=90); ax.set_ylabel(PRIMARY_METRIC)
    ax.set_title("25 configuraciones · " + PRIMARY_METRIC)
    plt.tight_layout(); fig.savefig(ARTIFACTS_DIR / "comparison.png", dpi=90); plt.show()
except Exception as exc:  # noqa: BLE001
    print("(gráfico omitido:", exc, ")")

## 18. Selección del modelo/configuración para la siguiente etapa

Se elige la mejor `f1_macro` en validation. **Sólo ahora** se evalúa el **TEST**, una única vez.

In [ ]:
test_pred = best_predictor.predict(X_test)
try: test_proba = best_predictor.predict_proba(X_test)[CLASSES].to_numpy()
except Exception: test_proba = None
test_metrics = compute_metrics(y_test, test_pred, test_proba, labels=CLASSES, prefix="test_")

selection = {"selected_config_id": BEST_CONFIG_ID, "primary_metric": PRIMARY_METRIC,
             "primary_metric_value": float(best_row[PRIMARY_METRIC]), "model_family": MODEL_FAMILY,
             "model_path": str(MODELS_DIR / BEST_CONFIG_ID), "dataset_version": DATASET_VERSION,
             "test_metrics": test_metrics}

# Registro del MEJOR modelo en el Model Registry de MLflow (lo que servirá la Inference API S6).
if USE_MLFLOW:
    try:
        best_run_id = RUN_IDS.get(BEST_CONFIG_ID)
        with mlflow.start_run(run_id=best_run_id):
            mlflow.log_metrics(test_metrics)             # métricas de test en el run ganador
            mlflow.set_tags({"selected": "true", "primary_metric": PRIMARY_METRIC})
            info = mlflow.pyfunc.log_model(
                artifact_path="model",
                python_model=AutogluonTabularModel(),
                artifacts={"predictor": str(MODELS_DIR / BEST_CONFIG_ID)},
                registered_model_name=MODEL_NAME,
                pip_requirements=["autogluon.tabular", "pandas", "scikit-learn"],
            )
        # Alias que carga la Inference API: models:/<MODEL_NAME>@<MODEL_ALIAS>
        try:
            from mlflow.tracking import MlflowClient
            mv = getattr(info, "registered_model_version", None)
            if mv is not None:
                MlflowClient().set_registered_model_alias(MODEL_NAME, MODEL_ALIAS, mv)
                selection["registered_model"] = {"name": MODEL_NAME, "version": int(mv), "alias": MODEL_ALIAS}
                print(f"Modelo registrado: {MODEL_NAME} v{mv} (alias '{MODEL_ALIAS}')")
        except Exception as exc:  # noqa: BLE001
            print(f"aviso: no se pudo fijar el alias '{MODEL_ALIAS}': {exc}")
    except Exception as exc:  # noqa: BLE001
        print(f"aviso: no se pudo registrar el modelo en MLflow: {exc}")

json.dump(selection, open(ARTIFACTS_DIR / "selected_model.json", "w"), indent=2)
print(json.dumps(selection, indent=2))

## 19. Exportación de resultados

In [ ]:
csv_path = RESULTS_DIR / "results.csv"
results_df.to_csv(csv_path, index=False); print("guardado:", csv_path)
try:
    results_df.to_parquet(RESULTS_DIR / "results.parquet", index=False)
    print("guardado:", RESULTS_DIR / "results.parquet")
except Exception as exc:  # noqa: BLE001
    print("(parquet omitido — instala pyarrow:", exc, ")")
print("\nartefactos:")
for p in sorted(ARTIFACTS_DIR.rglob("*")):
    if p.is_file(): print("  ", p.relative_to(ARTIFACTS_DIR))

## 20. Integración con MLflow (resumen)

La conexión con MLflow **ya está implementada** en este notebook (secciones 13b, 14 y 18):

- **Sección 13b:** se conecta a `MLFLOW_TRACKING_URI` y fija el experimento.
- **Sección 14:** cada una de las 25 configuraciones se registra como un *run* con
  `mlflow.start_run` → `log_params`, `log_metrics`, `set_tags` y el modelo como artefacto.
- **Sección 18:** el mejor modelo se publica en el **Model Registry** (`MODEL_NAME`) con el alias
  `MODEL_ALIAS` (`production`), que es el que carga la Inference API: `models:/<MODEL_NAME>@<MODEL_ALIAS>`.

Si MLflow no está disponible, el notebook igual corre y deja los payloads en
`artifacts/mlflow_payloads/`. La celda siguiente verifica qué quedó registrado.

In [ ]:
if USE_MLFLOW:
    try:
        runs = mlflow.search_runs(experiment_names=[MLFLOW_EXPERIMENT])
        print(f"runs registrados en '{MLFLOW_EXPERIMENT}':", len(runs))
        cols = [c for c in ["tags.config_id", "metrics.val_f1_macro", "metrics.test_f1_macro"] if c in runs.columns]
        display(runs[cols].head(30) if cols else runs.head())
    except Exception as exc:  # noqa: BLE001
        print("no se pudo consultar MLflow:", exc)
else:
    print("MLflow no activo. Payload de ejemplo (config ganadora), listo para registrar:")
    print(json.dumps(json.load(open(MLFLOW_DIR / f"{BEST_CONFIG_ID}.json")), indent=2))

## ENVIRONMENT

In [ ]:
import platform
env_info = {"python": platform.python_version(), "seed": SEED, "data_source": DATA_SOURCE,
            "mlflow_tracking_uri": MLFLOW_TRACKING_URI or "(no definido)", "mlflow_activo": USE_MLFLOW}
for mod in ["numpy", "pandas", "sklearn", "sqlalchemy", "autogluon", "lightgbm", "mlflow", "psycopg", "pyarrow"]:
    try: env_info[mod] = getattr(__import__(mod), "__version__", "?")
    except Exception: env_info[mod] = "no instalado"
json.dump(env_info, open(ARTIFACTS_DIR / "environment.json", "w"), indent=2)
print(json.dumps(env_info, indent=2))